# Költségtérítés elemzése

Ez a jegyzetfüzet bemutatja, hogyan lehet olyan ügynököket létrehozni, amelyek bővítményeket használnak helyi blokk képekből származó utazási költségek feldolgozására, költségtérítési e-mail generálására és költségadatok kördiagrammal való megjelenítésére. Az ügynökök dinamikusan választanak funkciókat a feladat kontextusa alapján.

Lépések:
1. Az OCR ügynök feldolgozza a helyi blokk képet és kinyeri az utazási költségadatokat.
2. Az E-mail ügynök generál egy költségtérítési e-mailt.

### Utazási költséghelyzet példája:
Képzeld el, hogy egy alkalmazott vagy, aki üzleti megbeszélés miatt utazik egy másik városba. A céged politikája szerint minden ésszerű utazással kapcsolatos költséget megtérítenek. Itt egy bontás a lehetséges utazási költségekről:
- Szállítás:
Oda-vissza repülőjegy az otthoni városodból az úti cél városába.
Taxi vagy utazásszervező szolgáltatások repülőtérre és onnan.
Helyi közlekedés az úti cél városában (például tömegközlekedés, bérautó vagy taxik).

- Szállás:
Három éjszakás szállodai tartózkodás egy középkategóriás üzleti szállodában, a találkozó helyszínéhez közel.

- Étkezések:
Napi étkezési juttatás reggelihez, ebédhez és vacsorához a cég per diem szabályzata alapján.

- Egyéb költségek:
Parkolási díjak a repülőtéren.
Internet-hozzáférési díjak a szállodában.
Borravalók vagy kisebb szolgáltatási díjak.

- Dokumentáció:
Minden blokkjegyzéket (repülőjegyek, taxik, szálloda, étkezések stb.) és egy kitöltött költségjelentést benyújtasz a térítéshez.


## Szükséges könyvtárak importálása

Importálja a jegyzetfüzethez szükséges könyvtárakat és modulokat.


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## Költségmodellek definiálása

 Hozzon létre egy Pydantic modellt az egyes költségekhez és egy ExpenseFormatter osztályt, amely a felhasználói lekérdezést strukturált költségadatokká alakítja.

 Minden költség a következő formátumban lesz ábrázolva:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Eszközök definiálása - E-mail generálása

Hozzon létre egy eszközfüggvényt, amely e-mailt generál egy költségtérítési igény benyújtásához.
- Ez az eszköz a Microsoft Agent Framework `@tool` dekorátorát használja.
- Kiszámolja a költségek összegét és az adatokat e-mail törzsbe formázza.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Utazási költségek kinyerésére szolgáló eszköz blokk képekből

Hozz létre egy eszköz függvényt az utazási költségek kinyerésére blokk képekből.
- Ez az eszköz a Microsoft Agent Framework `@tool` dekorátorát használja.
- Beolvassa a blokk képet, base64-ként kódolja, és visszaadja az adat URI-t az ügynök számára elemzés céljából.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Költségek feldolgozása

Határozd meg az ügynököket, és kösd össze őket egy szekvenciális munkafolyamatba a `WorkflowBuilder` használatával.
- Az OCR ügynök a `load_receipt_image` eszközt használva kinyeri a strukturált költségadatokat a nyugta képéből.
- Az Email ügynök a kinyert adatokat használva egy hivatalos költségtérítési e-mailt generál a `generate_expense_email` eszközzel.
- A `WorkflowBuilder` az `add_edge` segítségével egy szekvenciális folyamatot hoz létre: OCR ügynök → Email ügynök.


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Main function

Építse meg a szekvenciális munkafolyamatot, és futtassa le a blokkjegyzék kép feldolgozásához és a költségtérítési e-mail generálásához.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordító szolgáltatás [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár a pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások tartalmazhatnak hibákat vagy pontatlanságokat. Az eredeti dokumentum, annak anyanyelvén tekintendő hivatalos forrásnak. Kritikus információk esetén profi emberi fordítást javaslunk. Nem vállalunk felelősséget a fordítás használatából eredő félreértésekért vagy téves értelmezésekért.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
